<a href="https://colab.research.google.com/github/CleavDcos/Reasoning-LLMs-to-perform-Sentiment-Analysis/blob/main/Reasoning_LLMs_to_perform_Sentiment_Analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.Load the NEWS headline and description upon which to perform sentiment

2.Use deepseek(good reasoning model) to perform analaysis

3.Output not only sentiment(posi,neg,neutr) but also gives explanation

4.Present analysis in Gradio

In [4]:
print("Installing necessary libraries...")

# Transformers: Lets you easily use powerful pre-trained AI models for tasks like text generation, translation, or classification.
# Accelerate: Helps you run and train models efficiently across CPUs, GPUs.
# BitsAndBytes: Reduces the size of large models using quantization so they can fit and run on limited hardware like free GPUs.
# Torch (PyTorch): A deep learning library used to build, train, and run neural networks using tensors and GPU acceleration.

# Added 'datasets' to the list
!pip install -q transformers accelerate bitsandbytes torch datasets gradio

print("Libraries installed successfully!")

Installing necessary libraries...
Libraries installed successfully!


In [5]:
# Hugging Face Login

import os
from huggingface_hub import login, notebook_login

print("Attempting Hugging Face login...")

notebook_login()
print("Login successful (or token already present)!")

Attempting Hugging Face login...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Login successful (or token already present)!


In [6]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import load_dataset  # Import the dataset loading function
import gradio as gr
from IPython.display import display, Markdown
import random  # To pick random news items

# Check for GPU availability
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    torch.set_default_device("cuda")
    print("PyTorch default device set to CUDA (GPU).")
else:
    print("WARNING: No GPU detected. Performance will be very slow.")
    print("Go to Runtime > Change runtime type and select GPU.")


GPU detected: Tesla T4
PyTorch default device set to CUDA (GPU).


In [7]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown."""
    display(Markdown(text))

In [8]:
dataset_id = "ExponentialScience/DLT-Sentiment-News"
print(f"Loading dataset: {dataset_id}...")

news_dataset = load_dataset(dataset_id , split = "train")
print("dataset loaded successfully")

Loading dataset: ExponentialScience/DLT-Sentiment-News...


dataset loaded successfully


In [9]:
news_dataset

Dataset({
    features: ['timestamp', 'title', 'description', 'text', 'market_direction', 'engagement_quality', 'content_characteristics', 'vote_counts', 'total_votes', 'source_url', 'url', 'total_tokens'],
    num_rows: 23301
})

In [10]:
#lets view dataset as a pandas dataframe
news_dataset.to_pandas()

,timestamp,title,description,text,market_direction,engagement_quality,content_characteristics,vote_counts,total_votes,source_url,url,total_tokens
0,2022-06-09 04:00:43,Optimism Foundation sends $20M to the wrong wa...,The Optimism Foundation has issued a statement...,Optimism Foundation sends $20M to the wrong wa...,1,1,2,"{'bearish': 1, 'bullish': 0, 'liked': 3, 'disl...",4,https://cryptoslate.com/optimism-foundation-se...,https://cryptopanic.com/news/15442200/Optimism...,76
1,2022-06-09 03:31:47,75% of retailers eyeing crypto payments within...,"Improving customer experience, increasing the ...",75% of retailers eyeing crypto payments within...,2,1,1,"{'bearish': 0, 'bullish': 6, 'liked': 4, 'disl...",3,https://cointelegraph.com/news/75-of-retailers...,https://cryptopanic.com/news/15442145/75-of-re...,53
2,2022-06-09 01:22:23,Optimism loses 20M tokens after L1 and L2 conf...,Although the airdrop took place less than two ...,Optimism loses 20M tokens after L1 and L2 conf...,1,2,1,"{'bearish': 6, 'bullish': 0, 'liked': 3, 'disl...",2,https://cointelegraph.com/news/optimism-loses-...,https://cryptopanic.com/news/15441931/Optimism...,52
3,2022-06-08 23:00:19,U.S. Macro Pressure Responsible For Entire Bit...,There have been multiple sell-off trends recor...,U.S. Macro Pressure Responsible For Entire Bit...,1,2,2,"{'bearish': 2, 'bullish': 1, 'liked': 1, 'disl...",1,https://www.newsbtc.com/news/bitcoin/u-s-macro...,https://cryptopanic.com/news/15441620/US-Macro...,774
4,2022-06-08 22:20:11,Ethereum Ropsten Testnet Successfully Merged t...,"Ethereum's oldest testnet, Ropsten, was merged...",Ethereum Ropsten Testnet Successfully Merged t...,2,1,1,"{'bearish': 0, 'bullish': 5, 'liked': 2, 'disl...",2,https://blockworks.co/ethereum-ropsten-testnet...,https://cryptopanic.com/news/15441551/Ethereum...,50
...,...,...,...,...,...,...,...,...,...,...,...,...
23296,2025-05-13 20:40:00,Trump Coin Whales Dump $6.4M Ahead Of VIP Meme...,Two huge memecoin whales trigger multi-million...,Trump Coin Whales Dump $6.4M Ahead Of VIP Meme...,1,2,2,"{'bearish': 5, 'bullish': 0, 'liked': 0, 'disl...",8,https://dailycoin.com/trump-coin-whales-dump-6...,https://cryptopanic.com/news/21773506/Trump-Co...,50
23297,2025-05-06 22:30:00,Over 760K lost money on Donald Trump's memecoi...,"Over 764,000 wallets are in the red as Trump’s...",Over 760K lost money on Donald Trump's memecoi...,1,2,2,"{'bearish': 4, 'bullish': 0, 'liked': 1, 'disl...",6,https://www.thestreet.com/crypto/markets/over-...,https://cryptopanic.com/news/21606209/Over-760...,47
23298,2025-05-05 13:33:35,Trump to host memecoin gala dinner amid backla...,US President Donald Trump will host a gala din...,Trump to host memecoin gala dinner amid backla...,0,2,2,"{'bearish': 1, 'bullish': 3, 'liked': 1, 'disl...",3,https://cointelegraph.com/news/trump-s-stablec...,https://cryptopanic.com/news/21568390/Trump-to...,120
23299,2025-05-21 20:15:10,Vitalik Buterin praises real-time ZK proving m...,Ethereum co-founder Vitalik Buterin acknowledg...,Vitalik Buterin praises real-time ZK proving m...,1,0,2,"{'bearish': 1, 'bullish': 1, 'liked': 2, 'disl...",2,https://cryptoslate.com/vitalik-buterin-praise...,https://cryptopanic.com/news/21987380/Vitalik-...,67


In [11]:
#Now lets prepare the data for the LLM
#We wwill be combining title and description, and combining it to send it to LLm

def combine_news_text(example):
  #example here is one row of the dataset
  title = example.get("title", "") or ""
  description = example.get("description", "") or ""

  #Add seperator
  return {"full_text": f"Title: {title}\nDescription: {description}"}

In [12]:
#Apply the function to combine title and description into one full text
news_dataset = news_dataset.map(combine_news_text)

print("\n-----Sample data with full text----")
print(news_dataset[0])


-----Sample data with full text----
{'timestamp': '2022-06-09 04:00:43', 'title': 'Optimism Foundation sends $20M to the wrong wallet – OP drops 36%', 'description': 'The Optimism Foundation has issued a statement confirming that 20M OP tokens meant for a liquidity provisioning partner have been sent to the wrong address. The post Optimism Foundation sends $20M to the wrong wallet – OP drops 36% appeared first on CryptoSlate.', 'text': 'Optimism Foundation sends $20M to the wrong wallet – OP drops 36%\n\nThe Optimism Foundation has issued a statement confirming that 20M OP tokens meant for a liquidity provisioning partner have been sent to the wrong address. The post Optimism Foundation sends $20M to the wrong wallet – OP drops 36% appeared first on CryptoSlate.', 'market_direction': 1, 'engagement_quality': 1, 'content_characteristics': 2, 'vote_counts': {'bearish': 1, 'bullish': 0, 'liked': 3, 'disliked': 0, 'important': 1, 'lol': 3}, 'total_votes': 4, 'source_url': 'https://cryptos

In [13]:
# Let's display the full_text of the first sample
print(news_dataset[0]["full_text"])

Title: Optimism Foundation sends $20M to the wrong wallet – OP drops 36%
Description: The Optimism Foundation has issued a statement confirming that 20M OP tokens meant for a liquidity provisioning partner have been sent to the wrong address. The post Optimism Foundation sends $20M to the wrong wallet – OP drops 36% appeared first on CryptoSlate.


## ***LOAD AND TEST DEEPSEEK ***

In [14]:
pip install -U transformers


In [15]:
from transformers import BitsAndBytesConfig
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"loading model: {model_id}")
print("this may take a few minutes...")

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(model_id,
                             torch_dtype="auto",
                             quantization_config=bnb_config,
                             device_map="auto",
                             trust_remote_code=True)
model.eval()
print("model loaded successfully")

loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
this may take a few minutes...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

model loaded successfully


In [16]:
tokenizer=AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
print("Tokenizer loaded successfullyy")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfullyy


In [18]:

llm_pipeline = pipeline("text-generation", model = model, tokenizer = tokenizer, torch_dtype = "auto", device_map = "auto")
print("Text generation pipeline created successfully.")

`torch_dtype` is deprecated! Use `dtype` instead!


Text generation pipeline created successfully.


In [19]:
# Let's build the full formatted prompt:
# <|im_start|>user: Marks the beginning of the user's message.
# {test_question}: Inserts your question.
# <|im_end|>: Marks the end of the user’s message.
# <|im_start|>assistant: Signals that the assistant (AI) is about to reply.
# <think>: Encourages the AI to generate internal reasoning or thoughts before answering.

test_question = "Explain what electric cars are in simple terms. Keep the thinking short."
test_prompt = f"<|im_start|>user\n{test_question}<|im_end|>\n<|im_start|>assistant\n<think>\n"
print(f"Test prompt:\n{test_prompt}")

Test prompt:
<|im_start|>user
Explain what electric cars are in simple terms. Keep the thinking short.<|im_end|>
<|im_start|>assistant
<think>



In [20]:
#Test the model
# top_p: Nucleus sampling parameter (0.9 keeps responses on topic while allowing variety). Where the model chooses from the top 90% of likely next words (not just the top one). Helps keep output fluent but varied.
# do_sample: Enable sampling for more diverse outputs (vs greedy decoding). Enables sampling, meaning the model won’t always pick the highest probability next word — instead, it adds some randomness to make responses more creative and diverse.
outputs = llm_pipeline(
    test_prompt,
    max_new_tokens=4000,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)
outputs

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [21]:
# Function to extract and format the reason and output
#as we make the model think, we will seperate its thinking outputs and original output
def format_model_output(output):

    # Extract the content within <think> tags as Reason
    reason_start = output.find("<think>") + len("<think>")
    reason_end = output.find("</think>")
    reason = output[reason_start:reason_end].strip()

    # Extract the content after </think> as Output
    output_start = reason_end + len("</think>")
    model_output_content = output[output_start:].strip()

    # Format the result
    reason = f"**Reason**:\n{reason}\n"
    output = f"**Output**:\n{model_output_content}"
    return reason, output

In [22]:
# Extract and print the generated text (only the response part)
full_output = outputs[0]["generated_text"]
reason, output = format_model_output(full_output)
print_markdown(reason + output)

NameError: name 'outputs' is not defined

PROMPTING LLM FOR REASONING AND CLASSIFICATION

In [31]:
import re

def analyze_news_sentiment(news_text, llm_pipeline):
    """
    Analyzes news sentiment using the LLM, providing reasoning and classification.

    Args:
        news_text (str): The combined title and description of the news.
        llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.

    Returns:
        str: A string containing the reasoning and classification.
    """

    # Define the prompt template
    test_question = f"""You are a concise Financial News Analyst.
    Analyze the provided news text. Output ONLY the three requested items below, each on a new line, prefixed with the specified label.

    1.  REASONING: Brief financial reasoning (1-2 points max) for the sentiment.
    2.  SENTIMENT: Choose ONE: Positive, Negative, Neutral.
    3.  TAG: A concise topic tag (1-3 words).

    Do NOT add any other text, greetings, or explanations.

    News Text:
    {news_text}"""

    test_prompt = f"<|im_start|>user\n{test_question}<|im_end|>\n<|im_start|>assistant\n<think>\n"

    # Run model
    outputs = llm_pipeline(
        test_prompt,
        max_new_tokens=4000,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Extract the full generated text and parse it to separate reasoning from classification
    # The format_model_output function likely separates the thinking process from the final answer
    full_output = outputs[0]["generated_text"]
    reason, output = format_model_output(full_output)

    # Return both the reasoning process and the final sentiment classification
    return reason, output

In [32]:
# Print a separator line for clarity in the output
print("\n" + "=" * 30 + " TESTING ANALYSIS " + "=" * 30)

#select a few random indices from news dataset to test analysis function
random_indices=random.sample(range(len(news_dataset)),3)

#iterate over those news dataset
for index in random_indices:

  #Retrieve the full text of news items
  sample_news=news_dataset[index]["full_text"]

  reason,output = analyze_news_sentiment(sample_news,llm_pipeline)

  print_markdown(f"{reason}\n\n{output}")

  print("-"*60)

Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



============================== TESTING ANALYSIS ==============================


**Reason**:
Okay, so I'm trying to figure out what the user is asking for here. They provided some text, which is a news article about FTX CEO Sam Bankman-Fried. Then they asked me to analyze it and output three specific items. The first item is the reasoning, the second is the sentiment, and the third is the tag. They also gave an example of how the output should look.

First, I need to understand the structure of the news article. The title mentions that Sam Bankman-Fried is using cases to give crypto the bulk of its power. The description says that crypto billionaire is discussing use cases, and in an interview with NBA star Andre Iguodala, he outlines these use cases. So, the main points here are that crypto is becoming more powerful and that Sam is highlighting these use cases.

Now, for the analysis. The user wants the three items: reasoning, sentiment, and tag. The reasoning should be brief, about one to two points max. The sentiment is one of positive, negative, or neutral. The tag is a concise topic tag.

Looking at the news article, the key point is that crypto is becoming more powerful, and Sam is highlighting these use cases. So, for the sentiment, it's definitely positive because crypto is powerful. The reasoning could be something like, "Crypto is becoming more powerful, with crypto billionaire Sam Bankman-Fried highlighting these use cases, which contributes to its bulk power." That seems to cover it.

For the tag, I need something concise. Maybe "crypto" or "powerful." Either could work, but "powerful" is more specific.

Putting it all together, the output should have each item on a new line, prefixed with the label. So, starting with the label "REASONING," then "Sentiment," and "Tag." Each item is on a new line with the label before it.

I should make sure not to add any extra text or explanations, just stick to what the user provided. They emphasized not adding anything besides the three requested items. So, I'll avoid any other text or formatting.

Let me double-check to make sure I didn't miss anything. The label "REASONING" is correct, and the sentiment is positive. The tag is "powerful," which seems appropriate. The reasoning is clear and concise. The tags are concise and to the point. I think that covers it.

So, the final output should look like this:

REASONING
At its core, one of the things that Sam Bankman-Fried outlines in his interview is that crypto is becoming more powerful, which contributes to its bulk power.

Sentiment
Positive

Tag
powerful

Yes, that seems to fit perfectly. I don't think I missed any key points from the news article. The main idea is that crypto is powerful and that Sam is highlighting the use cases. So, the sentiment is positive, and the tag is "powerful," which accurately reflects the content of the article.


**Output**:
REASONING
At its core, one of the things that Sam Bankman-Fried outlines in his interview is that crypto is becoming more powerful, which contributes to its bulk power.

Sentiment
Positive

Tag
powerful

Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------


**Reason**:
Okay, so I need to analyze this news text and extract specific items. Let me start by reading the text carefully. The title is about XRP Ledger based exchange, Sologenic launching its new DEX. The description mentions Sologenic, an XRP Ledger tokenization ecosystem that facilitates investing and trading between crypto and non-blockchain assets. It also mentions the launch of Sologenic DEX.

First, I need to find the three requested items. The tags should be concise topics. The first tag is the topic itself, the second is the reasoning, and the third is the sentiment.

Looking at the description, the key points are the launch of Sologenic and its role in facilitating investments and trading between crypto and non-blockchain assets. So, the topic is "Investment & Trading", the reasoning is about facilitating crypto and non-blockchain asset trading, and the sentiment is positive.

Wait, but the description also mentions the tokenization aspect, which is a bit technical. But for the tags, I think the main focus is on the exchange launch and its role. So, tag: Investment & Trading, reasoning: Facilitate crypto and non-blockchain asset trading, sentiment: Positive.

I should make sure I'm not missing any other relevant points. The description doesn't mention other topics like finance or technology, so I think I'm okay. Let me double-check to ensure I'm not overcomplicating things.

Yes, I think that's correct. The tags are straightforward, and the sentiment is positive based on the exchange's success.


**Output**:
1. TAG: Investment & Trading
2. REASONING: Facilitate crypto and non-blockchain asset trading.
3. SENTiment: Positive

1. TAG: Investment & Trading
2. REASONING: Facilitate crypto and non-blockchain asset trading.
3. SENTiment: Positive

1. TAG: Investment & Trading
2. REASONING: Facilitate crypto and non-blockchain asset trading.
3. SENTiment: Positive

Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


------------------------------------------------------------


**Reason**:
Okay, so I'm trying to figure out what the user is asking for here. They provided a news text about Dubai's Crypto Regulator called VARA. The user wants me to analyze this news and extract three specific items: a tag, a reasoning, and a sentiment. Let me break this down.

First, I need to understand what the tag should be. The tag is a concise topic tag, so I should think of a short phrase that encapsulates the main point of the news. The news is about Dubai's crypto regulatory authority launching its metaverse debut. So a possible tag could be "Dubai Crypto Metaverse" or "Vara's Metaverse Debut."

Next, the reasoning. I should briefly explain the financial reasoning behind this. The reason is that the news highlights Dubai's crypto initiative, which is significant for investors. It shows the company's commitment to digital assets and the metaverse concept, which is a growing interest in finance.

Lastly, the sentiment. I need to choose one of the options: Positive, Negative, or Neutral. Looking at the news, it's all about Dubai's crypto initiatives and the metaverse, which are all positive. So the sentiment here would be Positive.

I should make sure not to add any extra text or explanations. The user specified to do only that. So, I'll focus on extracting the three items: tag, reasoning, and sentiment. I'll choose the most appropriate tag, the shortest reasoning, and the positive sentiment.

Let me double-check the analysis. The tag "Vara's Metaverse Debut" seems accurate. The reasoning is concise and directly ties into the main points of the news. The sentiment is definitely Positive because it's highlighting Dubai's crypto initiatives, which are positive in the financial landscape.

I think that's it. I'll present the three items as requested, using the tag "Vara's Metaverse Debut," the reasoning about the significance of the initiative, and the sentiment being Positive.


**Output**:
1. TAG: Vara's Metaverse Debut  
2. REASONING: The news highlights Dubai's crypto initiative, which is significant for investors. It shows the company's commitment to digital assets and the metaverse concept, which is a growing interest in finance.  
3. SENTiment: Positive

------------------------------------------------------------


BUILD THE GRADIO INTERFACE

In [ ]:
import gradio as gr  # Importing the Gradio library for creating the web interface
import random  # Importing the random library to generate random numbers
import re  # Importing the regular expressions library (not used in this snippet)
from transformers import pipeline  # Importing the pipeline function from transformers for model loading

# --- Gradio Helper Functions ---
def get_random_news():
    """Fetches and returns the full_text of a random news item."""
    if not news_dataset:  # Check if the news dataset is empty
        return "Error: No news items available."  # Return an error message if no news items are found
    random_index = random.randint(0, len(news_dataset) - 1)  # Generate a random index to select a news item
    news_text = news_dataset[random_index]['full_text']  # Retrieve the full text of the news item at the random index
    print(f"Fetched news item at index {random_index}")  # Print the index of the fetched news item
    return news_text  # Return the fetched news text

def perform_analysis(news_text):
    """Triggers analysis on the provided news text."""
    if not news_text or news_text.startswith("Error"):  # Check if the news text is empty or an error message
        return "Error: No news text to analyze."  # Return an error message if no valid news text is provided
    print(f"Analyzing news: {news_text[:50]}...")  # Print the first 50 characters of the news text being analyzed
    reason, output = analyze_news_sentiment(news_text, llm_pipeline)  # Analyze the sentiment of the news text
    return reason, output  # Return the reasoning and output from the analysis

# --- Build Gradio Interface ---
with gr.Blocks(theme=gr.themes.Glass()) as demo:  # Create a Gradio Blocks interface with a glass theme
    gr.Markdown("""
    # DeepSeek Financial News Analyzer
    Fetches a random news item from the dataset.
    Click 'Analyze News' to get sentiment classification and the model's reasoning.

    """)

    with gr.Row():  # Create a row for buttons
        btn_fetch = gr.Button("🔄 Fetch Random News Item")  # Button to fetch a random news item
        btn_analyze = gr.Button("💡 Analyze News", variant="primary")  # Button to analyze the news

    news_display = gr.Textbox(  # Textbox to display the news item
        label="📰 News Text (Title & Description)",  # Label for the textbox
        lines=8,  # Number of lines in the textbox
        interactive=False,  # Make the textbox non-interactive
        placeholder="Click 'Fetch Random News Item' to display news."  # Placeholder text
    )
    # Creates a collapsible panel
    with gr.Accordion("🤖 Model Reason", open=True):  # Accordion for model reasoning
        analysis_display = gr.Markdown()  # Markdown display for the reasoning

    with gr.Accordion("🤖 Model Output", open=True):  # Accordion for model output
        analysis_output = gr.Markdown()  # Markdown display for the output

    # --- Event Handlers ---
    btn_fetch.click(  # Set up click event for the fetch button
        fn=get_random_news,  # Function to call when the button is clicked
        inputs=[],  # No inputs needed
        outputs=[news_display]  # Output to the news display textbox
    )

    btn_analyze.click(  # Set up click event for the analyze button
        fn=perform_analysis,  # Function to call when the button is clicked
        inputs=[news_display],  # Input from the news display textbox
        outputs=[analysis_display, analysis_output]  # Outputs to the reasoning and output displays
    )

    # Load initial news item when the app starts
    demo.load(  # Load function to run when the app starts
        fn=get_random_news,  # Function to call to get a random news item
        inputs=None,  # No inputs needed
        outputs=[news_display]  # Output to the news display textbox
    )

# --- Launch the Gradio App ---
print("Launching Gradio demo...")  # Print message indicating the app is launching
demo.launch(debug=True)  # Launch the Gradio app in debug mode

/tmp/ipykernel_8697/1399811874.py:25: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Glass()) as demo:  # Create a Gradio Blocks interface with a glass theme


Launching Gradio demo...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://19e7d3a1429f5220b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Fetched news item at index 2346
Fetched news item at index 18939


Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title: Cardano on Verge of "Most Significant" Mile...
Fetched news item at index 13797
